# Persian News Recommender System

**Content-based article recommendations with Hazm, TF-IDF and cosine similarity**

Given a Persian news article, this notebook finds the five most similar articles in a corpus of 5,000 news items spanning eight categories. It is a classic content-based recommender: no user history is needed, only the text of the articles.

### Pipeline

| Stage | Tooling | What happens |
|---|---|---|
| 1. Preprocessing | Hazm | Normalize, tokenize, lemmatize, remove stop words |
| 2. Vectorization | scikit-learn `TfidfVectorizer` | Turn each article into a 5,000-dimensional TF-IDF vector |
| 3. Similarity | scikit-learn `cosine_similarity` | Compute pairwise similarity between all articles |
| 4. Recommendation | NumPy | Rank by similarity and return the top 5 |

### Data

5,000 news articles published by the Young Journalists Club (YJC), Iran, in eight categories: culture & art, science & technology, politics, social, medical, international, economy and sports. The dataset is not included in this repository (see the README).

## 1. Setup

Install the dependencies with `pip install -r requirements.txt`. [Hazm](https://github.com/roshan-research/hazm) is an open-source NLP toolkit for Persian, and it handles the language-specific parts of the pipeline.

In [1]:
import re

import numpy as np
import pandas as pd
from hazm import Lemmatizer, Normalizer, WordTokenizer, stopwords_list
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def preview(text, n=300):
    """Print the first n characters of a text, to keep notebook output short."""
    print(text[:n] + ("..." if len(text) > n else ""))

## 2. Load the data

Each row is one article: its full text (`content`), its category name (`label`) and a numeric category id (`label_id`). The CSV is expected at `data/news_data.csv`.

In [2]:
df = pd.read_csv("../data/news_data.csv")
df.head()

,content,label,label_id
0,به گزارش حوزه ادبیات گروه فرهنگی باشگاه خبرنگا...,فرهنگی هنری,5
1,به گزارش خبرنگار دانشگاه باشگاه خبرنگاران، نما...,سیاسی,3
2,به گزارش حوزه تحولات منطقه گروه سیاسی باشگاه خ...,سیاسی,3
3,به گزارش خبرنگار حوزه رادیو تلویزیون گروه فرهن...,فرهنگی هنری,5
4,واکنش‌ها نسبت به لغو حضور مدافع سابق تیم ملی ف...,سیاسی,3


## 3. Exploratory analysis

Before processing anything, check the size and schema of the data, missing values, class balance and duplicates.

In [3]:
print(df.shape)
print(df.columns.tolist())

(5000, 3)
['content', 'label', 'label_id']


In [4]:
# a few random articles (fixed seed so the sample is reproducible)
df.sample(5, random_state=42)

,content,label,label_id
1501,به گزارش خبرنگار حوزه رفاه و تعاون گروه اجتماع...,اجتماعی,0
2586,به گزارش خبرنگار ورزشی باشگاه خبرنگاران، اردوی...,ورزشی,6
2653,به گزارش خبرنگار ورزشی باشگاه خبرنگاران؛ اولین...,ورزشی,6
1055,به گزارش حوزه دانشگاه باشگاه خبرنگاران، محمد ا...,سیاسی,3
705,به گزارش خبرنگار پارلمانی باشگاه خبرنگاران، نم...,سیاسی,3


In [5]:
df.info()
df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   content   5000 non-null   str  
 1   label     5000 non-null   str  
 2   label_id  5000 non-null   int64
dtypes: int64(1), str(2)
memory usage: 117.3 KB


content     0
label       0
label_id    0
dtype: int64

In [6]:
df["label"].value_counts()

label
فرهنگی هنری    802
علمی فناوری    770
سیاسی          688
اجتماعی        665
پزشکی          624
بین الملل      580
اقتصادی        443
ورزشی          428
Name: count, dtype: int64

In [7]:
# exact duplicate articles
df.duplicated(subset="content").sum()

np.int64(0)

In [8]:
# first 300 characters of a raw article
preview(df["content"].iloc[0])

به گزارش حوزه ادبیات گروه فرهنگی باشگاه خبرنگاران جوان به نقل از اداره کل روابط عمومی و امور ‌بین‌الملل کانون، نمایشگاه کتاب بولونیا در حالی روز پایانی خود را سپری می‌کند که یکی از آثار ماندگار ادبیات کودک ایران برای نخستین بار به زبان انگلیسی در آن معرفی شده است. تصویرگری این کتاب را آلن بایاش و تر...


**Findings**

- The corpus has 5,000 articles and no missing values or exact duplicates.
- The eight categories are reasonably balanced, from 428 articles (sports) to 802 (culture & art), so no resampling is needed.
- Articles come from a single news agency and share a template ("به گزارش ... باشگاه خبرنگاران جوان", "انتهای پیام"). This boilerplate matters later, because shared phrases can create similarity that has nothing to do with the topic.

## 4. Text preprocessing

Persian needs language-specific handling: character variants, the half-space (ZWNJ, U+200C), rich verb morphology and a different stop word list. The pipeline below is applied to a **copy** of the data, so the raw articles stay available for display.

1. **Normalize** – Hazm's `Normalizer` unifies character variants and fixes spacing and half-space usage.
2. **Remove punctuation and digits** – done with regular expressions. The half-space is deliberately kept, because stripping it would break words such as «می‌روم».
3. **Tokenize** – split the text into words with Hazm's `WordTokenizer`.
4. **Lemmatize** – reduce inflected forms to a base form. For verbs Hazm returns a `past#present` pair, for example `کرد#کن`.
5. **Remove stop words** – drop very frequent function words using Hazm's stop word list.

In [9]:
data = df.copy()
data.head()

,content,label,label_id
0,به گزارش حوزه ادبیات گروه فرهنگی باشگاه خبرنگا...,فرهنگی هنری,5
1,به گزارش خبرنگار دانشگاه باشگاه خبرنگاران، نما...,سیاسی,3
2,به گزارش حوزه تحولات منطقه گروه سیاسی باشگاه خ...,سیاسی,3
3,به گزارش خبرنگار حوزه رادیو تلویزیون گروه فرهن...,فرهنگی هنری,5
4,واکنش‌ها نسبت به لغو حضور مدافع سابق تیم ملی ف...,سیاسی,3


In [10]:
lemmatizer = Lemmatizer()
normalizer = Normalizer()
tokenizer = WordTokenizer()
stop_words = set(stopwords_list())


def persian_text_preprocessing(text):
    """Normalize, clean, tokenize, lemmatize and filter a Persian text.

    Returns a single space-separated string, ready for TF-IDF.
    """
    if not isinstance(text, str):
        return ""

    normalized_text = normalizer.normalize(text)

    # drop punctuation but keep the half-space (\u200c), then drop digits
    regexed_text = re.sub(r'[^\w\s\u200c]', ' ', normalized_text)
    regexed_text = re.sub(r'\d+', ' ', regexed_text)

    tokenized_text = tokenizer.tokenize(regexed_text)
    lemmatized_text = [lemmatizer.lemmatize(word) for word in tokenized_text]
    result = [word for word in lemmatized_text if word not in stop_words]

    return " ".join(result)

In [11]:
data["content"] = data["content"].apply(persian_text_preprocessing)

In [12]:
# the same article as above, after preprocessing
preview(data["content"].iloc[0])

گزارش حوزه ادبیات گروه فرهیخت#فرهنگ باشگاه خبرنگار جوان نقل اداره روابط عمومی امور بین‌الملل کانون نمایشگاه کتاب بولونیا روز پایانی سپری کرد#کن آثار ماندگار ادبیات کودک ایران زبان انگلیسی معرفی‌شده تصویرگری کتاب آلن بای ترجمه میترا خاتون‌آباد داشت#دار بابابرفی داستان کودکانی کمک آدم‌برفی شبیه پدربزر...


## 5. Building the document profile

Each article is represented by a *tag*: its category label followed by its preprocessed text. Prepending the label adds the category name as an extra term, which nudges similarity towards articles from the same section.

This is a design choice with a trade-off: it feeds supervised information (the label) into the representation. Any category-level accuracy measured on these vectors would therefore be optimistic (see the limitations at the end).

In [13]:
data["tag"] = data["label"] + " " + data["content"]

## 6. TF-IDF vectorization

TF-IDF gives a term a high weight when it is frequent in one article but rare across the corpus. Generic words are down-weighted and distinctive words stand out. The vocabulary is capped at the 5,000 highest-frequency terms (`max_features=5000`) to keep the representation compact. The result is a sparse matrix with one row per article.

In [14]:
tfidf = TfidfVectorizer(max_features=5000)
tfidf_matrix = tfidf.fit_transform(data["tag"])

In [15]:
print(tfidf_matrix.shape)    # (n_articles, vocabulary_size)

(5000, 5000)


## 7. Cosine similarity

Cosine similarity measures the angle between two vectors: 1 means the same direction (identical term profile) and 0 means nothing in common. TF-IDF weights are non-negative, so all scores fall between 0 and 1.

The full pairwise matrix is computed once. Row *i* holds the similarity of article *i* to every article in the corpus. A 5,000 × 5,000 matrix of float64 values takes about 200 MB, which is fine here but does not scale to much larger corpora (see the limitations).

In [16]:
similarity_matrix = cosine_similarity(tfidf_matrix)

In [17]:
print(similarity_matrix.shape)    # (n_articles, n_articles)
print(similarity_matrix[0, 0])    # the diagonal is 1: an article is identical to itself

(5000, 5000)
1.0000000000000004


## 8. The recommender

`find_similar_news` takes the similarity row of the query article, sorts it from most to least similar, removes the article itself (it always has similarity 1 with itself) and returns the indices of the top results. `show_recommendations` prints the query and its recommendations with category and score, so results are easy to judge by eye.

In [18]:
def find_similar_news(news_index, top_n=5):
    """Return the indices of the top_n articles most similar to the given article."""
    scores = similarity_matrix[news_index]

    # indices sorted from highest to lowest similarity
    sorted_idx = np.argsort(scores)[::-1]

    # the article is always most similar to itself, so exclude it
    sorted_idx = [i for i in sorted_idx if i != news_index]

    return sorted_idx[:top_n]


def show_recommendations(news_index, top_n=5):
    """Print an article and its top_n most similar articles."""
    print(f"QUERY: article #{news_index} [{df['label'].iloc[news_index]}]")
    preview(df["content"].iloc[news_index])

    print("\nRECOMMENDATIONS")
    for rank, idx in enumerate(find_similar_news(news_index, top_n), start=1):
        score = similarity_matrix[news_index, idx]
        print(f"\n{rank}. article #{idx} [{df['label'].iloc[idx]}] (cosine similarity: {score:.3f})")
        preview(df["content"].iloc[idx])

## 9. Qualitative evaluation

There is no ground truth for "similar articles", so the first check is to read the recommendations for a few articles and judge whether they are topically related.

In [19]:
show_recommendations(2)

QUERY: article #2 [سیاسی]
به گزارش حوزه تحولات منطقه گروه سیاسی باشگاه خبرنگاران جوان؛ نیروهای ارتش سوریه همزمان با تامین امنیت هزاران غیر نظامی محاصره شده توسط گروه‌های تروریستی جهت خروج از غوطه شرقی، شهرک حموریه را نیز آزاد کردند. نیروهای ارتش و هلال احمر سوریه امنیت هزاران شهروند محاصره شده در غوطه شرقی را برای خروج از آن...

RECOMMENDATIONS

1. article #3659 [بین الملل] (cosine similarity: 0.552)
به گزارش گروه بین الملل باشگاه خبرنگاران جوان به نقل از شبکه تلویزیونی العالم؛ نیروهای ارتش سوریه در شرق حلب به پیشروی‌های قابل ملاحظه‌ای دست یافتند. بر این اساس، ارتش سوریه و نیروهای هم پیمان در ادامه عملیات نظامی خود در محورهای باب النیرب و الصالحین پس از شکست دادن تروریست‌ها به پیشروی‌های قابل مل...

2. article #2838 [بین الملل] (cosine similarity: 0.458)
به گزارش سرویس بین‌ الملل باشگاه خبرنگاران به نقل از شبکه تلویزیونی سوریه؛ ارتش این کشور توانست پس از هدف قرار دادن همه مراکز و کانونهای تجمع تروریستها در این منطقه آنرا پاکسازی کند و امنیت را به آن بازگرداند. گزارشگر شبکه تلویزیونی سو

In [20]:
show_recommendations(10)

QUERY: article #10 [اجتماعی]
سید مجتبی نجاتی در گفتگو با خبرنگار اجتماعی باشگاه خبرنگاران گفت: برگزاری اردوها و تورهای آموزشی و توانمندسازی شهروندان در محلات ۱۳گانه منطقه ۱۹ در دستورکار اداره فرهنگی شهرداری این منطقه قرار دارد. وی افزود: این طرح همزمان با هفته سلامت و روز جهانی بهداشت از یکم تا هفتم اردیبهشت ماه سال ۹۳ در دست ا...

RECOMMENDATIONS

1. article #3433 [اجتماعی] (cosine similarity: 0.635)
مهناز اسکندر مدیر اداره آموزش شهروندی شهرداری منطقه ۱۲ در گفتگو با خبرنگار اجتماعی باشگاه خبرنگاران گفت: برگزاری آموزش چهره به چهره آپارتمان نشینی در محلات ۱۳ گانه منطقه ۱۲ همزمان با فصل تابستان در دستور کار اداره آموزش شهروندی شهرداری این منطقه قرار دارد. وی ادامه داد: این طرح همزمان با فصل تابستان...

2. article #4250 [اجتماعی] (cosine similarity: 0.564)
محمدرضا یاوری مدیر اداره سلامت شهرداری منطقه ۱۵ در گفتگو باخبرنگار اجتماعی باشگاه خبرنگارانگفت: اجرای طرح‌های سلامت و بهداشت قلب کودکان همزمان با هفته سلامت قلب کودکان در دستور کار اداره سلامت شهرداری منطقه ۱۵ قرار دارد. وی ادامه داد: ا

**Observations**

- For article #2, a report on Syrian army operations in Eastern Ghouta, the recommended articles are all other reports on the Syrian conflict.
- For article #10, a news item about activities of a Tehran municipality district, the results are other district-level municipal news. Part of that similarity comes from the shared agency template and district vocabulary, and not only from the topic. Some recommendations are therefore closer in style than in subject, which is the main weakness of a purely lexical approach.

## 10. Example: tabular output for article #46

The recommendations can also be returned as a small table with the category and index of each article, which is handy for downstream use.

In [21]:
top5_idx = find_similar_news(46)

recommendations = pd.DataFrame({
    "label": df["label"].iloc[top5_idx].values,
    "index": top5_idx,
})
recommendations

,label,index
0,علمی فناوری,1688
1,علمی فناوری,4844
2,علمی فناوری,2578
3,علمی فناوری,2048
4,علمی فناوری,3142


All five recommendations for article #46 belong to the science & technology category.

## Limitations and next steps

- **Lexical, not semantic.** TF-IDF matches words, not meaning, so synonyms and paraphrases are missed. A natural next step is to compare against sentence embeddings from a Persian or multilingual transformer (for example ParsBERT or a multilingual sentence-transformers model).
- **Agency boilerplate.** Template phrases shared by every article inflate similarity. Corpus-specific stop words or a `max_df` threshold in `TfidfVectorizer` would reduce this.
- **Label in the representation.** The tag includes the category, so same-category matches are partly built in. Comparing against vectors built from the text only would show how much it helps.
- **No quantitative evaluation yet.** Metrics such as precision@k of the category, or human relevance judgments, would complement the manual inspection above.
- **Scalability.** The dense n × n similarity matrix grows quadratically. For larger corpora, compute top-k neighbours with sparse nearest-neighbour search or an approximate index such as FAISS.
- **No personalization.** This is a pure content-based approach. Adding user interaction data would enable collaborative or hybrid recommendations.